In [12]:
import numpy as np
import matplotlib.pyplot as plt


In [18]:
from Pycrafter6500 import pycrafter6500 as pyc


pattern = np.zeros((1080, 1920), dtype=np.uint8)
pattern[:, :960] = 1  # Set the left half to 1 

images = [pattern]

# Instantiate the DMD
dlp = pyc.dmd()
dlp.stopsequence()
dlp.changemode(3) # Pattern Sequence Mode

exposure = [1000000] # 1 second
dark_time = [0]
trigger_in = [False]
trigger_out = [1]

dlp.defsequence(images, exposure, trigger_in, dark_time, trigger_out, 0)
dlp.startsequence()



merging...
encoding...
uploading...
0 13


In [ ]:
from src import superpixel as px
from src import propagation as prop
from src import helpers
from src import models 
from src import plotting

device = "mps"

wavelength = 473e-9 # m


# DMD parameters 
mirror_pitch = 7.56e-6
mirror_size_px = (1080, 1920)  # (ny, nx) = (vert, horiz)
mirror_size_m = (mirror_pitch * mirror_size_px[0], mirror_pitch * mirror_size_px[1])  # (y, x) in meters


superpixel_dimension = 3 
superpixel_size_m = (superpixel_dimension * mirror_pitch, superpixel_dimension * mirror_pitch)

# relay parameters
f1 = 0.3  # m
f2 = 0.15  # m

# Note on Lens 1 clear aperture:
# Light from a mirror at x0 reaching the aperture crosses lens 1 at x0 + aperture_position.
# From the furthest DMD corner, this is up to ~16 mm off-axis, which exceeds a 1-inch lens
# radius (12.7 mm). Use None (unclipped) or >= 2-inch lens (25.4e-3 m) to avoid corner dimming.
lens1_clear_radius = None  # 12.7e-3 m (1-inch) will clip DMD corners

L = wavelength * f1 / mirror_pitch  # m

aperture_y = -L / superpixel_dimension       
aperture_x = -L / (superpixel_dimension**2)  
aperture_position = (aperture_y, aperture_x) 


max_aperture_radius = L / (4 * superpixel_dimension)  # m
real_aperture_radius = 0.75 * max_aperture_radius     # chosen arbitrarily

print(max_aperture_radius)
print(px.pixel_phases(superpixel_dimension))


samples_per_mirror = 2
num_pix = 2 * mirror_size_px[1] * samples_per_mirror  
grid_shape = (num_pix, num_pix)  # (ny, nx)

dy = mirror_pitch / samples_per_mirror
dx = mirror_pitch / samples_per_mirror
grid_spacing = (dy, dx)  # (dy, dx) in meters for propagation.py
grid_extent_m = (num_pix * dy, num_pix * dx)  # (y, x) in meters



0.001564153439153439
[[0.         0.6981317  1.3962634 ]
 [2.0943951  2.7925268  3.4906585 ]
 [4.1887902  4.88692191 5.58505361]]


In [27]:
target = helpers.generate_plane_wave(mirror_size_px, superpixel_dimension)



# encode
commands, fields = px.build_superpixel_fields(superpixel_dimension, px.pixel_phases(superpixel_dimension))
dmd_commands = px.encode(target, superpixel_dimension, commands, fields) 
print(dmd_commands.shape)
# plotting.plot_dmd_actuators(commands, 
#                    superpixel_dimension=3,
#                    save_path="jupyter_commands.png")
# dmd_commands = 1-dmd_commands

(360, 640, 9)


In [28]:
mirror_image = px.to_image(dmd_commands, superpixel_dimension)        
images = [mirror_image]


exposure = [1000000]  # 1 second
dark_time = [0]
trigger_in = [False]
trigger_out = [1]
dlp.defsequence(images, exposure, trigger_in, dark_time, trigger_out, 0)
dlp.startsequence()

merging...
encoding...
uploading...
0 3667
100 3667
200 3667
300 3667
400 3667
500 3667
600 3667
700 3667
800 3667
900 3667
1000 3667
1100 3667
1200 3667
1300 3667
1400 3667
1500 3667
1600 3667
1700 3667
1800 3667
1900 3667
2000 3667
2100 3667
2200 3667
2300 3667
2400 3667
2500 3667
2600 3667
2700 3667
2800 3667
2900 3667
3000 3667
3100 3667
3200 3667
3300 3667
3400 3667
3500 3667
3600 3667


In [ ]:
# generate a sample target function
target = helpers.generate_LG10_donut(mirror_size_px, waist=60, superpixel_dimension=superpixel_dimension)

# encode
commands, fields = px.build_superpixel_fields(superpixel_dimension, px.pixel_phases(superpixel_dimension))
dmd_commands = px.encode(target, superpixel_dimension, commands, fields)         # (360, 640, 9)
dmd_commands = 1-dmd_commands
helpers.plot_dmd_actuators(commands, 
                   superpixel_dimension=3,
                   save_path="jupyter_commands.png")

In [11]:
mirror_image = px.to_image(dmd_commands, superpixel_dimension)        
images = [mirror_image]


exposure = [1000000]  # 1 second
dark_time = [0]
trigger_in = [False]
trigger_out = [1]
dlp.defsequence(images, exposure, trigger_in, dark_time, trigger_out, 0)
dlp.startsequence()

merging...
encoding...
uploading...
0 1757
100 1757
200 1757
300 1757
400 1757
500 1757
600 1757
700 1757
800 1757
900 1757
1000 1757
1100 1757
1200 1757
1300 1757
1400 1757
1500 1757
1600 1757
1700 1757


In [ ]:
# stop the sequence
dlp.stopsequence()